In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms,datasets
import wandb
from key import KEY
import torch.nn.functional as F
import os
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

### 코드 계획 
1. 이미지를 하나 가져오기
2. 기존 방법 FCNN 사용해서 그 형태를 보여주기기
3. 기존 방법 FCNN 사용해서 예측을 시도
4. cnn 구현 및 형태 샘플 보여주기
5. cnn을 가지고 예측을 시도
6. 결과를 비교



In [2]:
img = Image.open('data/img.jpg')

In [3]:
wandb.login(key=KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\최연호\_netrc
wandb: Currently logged in as: chldusgh0497 (chldusgh0497-jeonbuk) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [5]:


class SimpleFCNN(nn.Module):
    def __init__(self, input_size=64, hidden_sizes=[512, 256, 128], num_classes=2):
        super(SimpleFCNN, self).__init__()
        
        # 입력 크기 계산 (RGB 이미지)
        self.input_features = input_size * input_size * 3
        
        # 은닉층들
        layers = []
        prev_size = self.input_features
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            prev_size = hidden_size
        
        # 출력층
        layers.append(nn.Linear(prev_size, num_classes))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        # 이미지를 1차원으로 평탄화
        x = x.view(x.size(0), -1)
        return self.network(x)

In [6]:
# train_loader= DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
# val_loader= DataLoader(val_dataset, batch_size=128, shuffle=True, num_workers=0)
# test_loader= DataLoader(test_dataset, batch_size=128, shuffle=True, num_workers=0)

In [7]:
def data_load(image_size=64, batch_size=128):
    train_dir = 'data/archive/training_set/training_set'
    val_dir = 'data/archive/validation_set/validation_set'
    test_dir = 'data/archive/test_set/test_set'
    trans= transforms.Compose([transforms.Resize((image_size, image_size)), 
                           transforms.ToTensor(), 
                           transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])])


    train_dataset=datasets.ImageFolder(root=train_dir, transform=trans)
    val_dataset=datasets.ImageFolder(root=val_dir, transform=trans)
    test_dataset=datasets.ImageFolder(root=test_dir, transform=trans)
    train_loader= DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader= DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader= DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    return train_loader, val_loader, test_loader

In [8]:
# dataiter = iter(train_loader)

# # 이터레이터에서 첫 번째 배치의 데이터를 가져옵니다
# images, labels = next(dataiter)

# print(images.shape)
# print(labels.shape)
# torch.Size([128, 3, 64, 64]) 배치사이즈. 색상수, 그림 높이, 그림 너비

In [9]:
class convolution(nn.Module):
    def __init__(self, conv1_out=32, conv2_out=64, fc1_out=128, fc2_out=10, input_size=64):
        super(convolution, self).__init__()
        self.conv1_size = conv1_out
        self.conv2_size = conv2_out
        self.fc1_size = fc1_out
        self.fc2_size = fc2_out

        self.conv1 = nn.Conv2d(3, conv1_out, 3)
        self.conv2 = nn.Conv2d(conv1_out, conv2_out, 3)
        self.pool = nn.MaxPool2d(2, 2)

        # fc1은 forward에서 동적으로 생성
        self.fc1 = None
        self.fc2 = nn.Linear(fc1_out, fc2_out)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        # 동적으로 특성 수 계산
        batch_size = x.size(0)
        features = x.size(1) * x.size(2) * x.size(3)
        x = x.view(batch_size, -1)
        
        # fc1을 동적으로 생성 (첫 번째 실행 시)
        if self.fc1 is None:
            self.fc1 = nn.Linear(features, self.fc1_size).to(x.device)
        
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [10]:
def train_model(model,device, train_loader, val_loader, criterion, optimizer, epochs=10):
    


    for epoch in range(epochs):
        # print(f'Starting Epoch: {epoch+1}...')
        running_loss = 0.0
        model.train()
        model.to(device)

        for i, data in enumerate(train_loader, 0):
            inputs, labels = data

            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
        

        model.eval()
        with torch.no_grad():
            for i, data in enumerate(val_loader, 0):
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_loss += loss.item()
        

        
    return model
    






In [11]:
def evaluate_model(model,device, test_loader):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            # 예측값과 실제값 저장
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    accuracy = 100 * correct / total
    conf_matrix = confusion_matrix(all_labels, all_preds)

    return accuracy, conf_matrix

In [12]:

def process_model(learning_rate=0.001, image_size=64, batch_size=128, hidden_size_list=[32,64,128],  epochs=10, i=0):
    #데이터 호출
    wandb.init(project="cnn_test_img_265", name=f"fcnn_{i}",config={
        "learning_rate": learning_rate, 
        "image_size": image_size,
        "batch_size": batch_size, 
        "epochs": epochs,
        "hidden_size_list": hidden_size_list,
        })
    device= torch.device("cuda" if torch.cuda.is_available() else "cpu")

    
    train_loader, val_loader, test_loader= data_load(image_size, batch_size)
    hidden_size_list.reverse()
    # model= convolution(conv1_out=hidden_size_list[0], conv2_out=hidden_size_list[1], fc1_out=hidden_size_list[2], fc2_out=2, input_size=image_size)
    model =SimpleFCNN(input_size=image_size, hidden_sizes=hidden_size_list, num_classes=2)
    #optimizer 선택
    optimizer= optim.Adam(model.parameters(), lr=learning_rate)
    criterion= nn.CrossEntropyLoss()

    #모델 생성
    #모델 학습
    model= train_model(model=model,device=device, train_loader=train_loader, val_loader=val_loader, criterion=criterion, optimizer=optimizer, epochs=epochs)
    
    #모델평가
    accuracy, conf_matrix= evaluate_model(model=model, device=device, test_loader=test_loader)
    wandb.log({"accuracy": accuracy, "conf_matrix": conf_matrix})
    wandb.finish()

In [13]:
# cnn_6
params = {
    "learning_rate": [0.001],
    "image_size": [128],
    "batch_size": [64],
    "epochs": [30],
    "hidden_sizes": [[128,256,512]],
}

In [ ]:
for i in range(5):
    for learning_rate in params['learning_rate']:
       for image_size in params['image_size']:
           for batch_size in params['batch_size']:
               for epochs in params['epochs']:
                   for hidden_size_list in params['hidden_sizes']:
                       print(f'Starting process: {i}...')
                       process_model(learning_rate=learning_rate, image_size=image_size, batch_size=batch_size, epochs=epochs, hidden_size_list=hidden_size_list, i=i)





Starting process: 0...
